<a href="https://colab.research.google.com/github/sabithakrishnan/fall_detection/blob/main/detection_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import torch
import torchvision.transforms as T
from PIL import Image


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torch.hub.load('facebookresearch/detr', 'detr_resnet101', pretrained=True)
model.eval()
model.to(device)

#assign person class as 1
PERSON_CLASS_IDX = 1

#image transformations
transform = T.Compose([
    T.Resize(800),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Helper function to convert DETR bounding boxes (center x, center y, width, height) to absolute pixels
def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(-1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=-1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32, device=device)
    return b

Open Video Source
video_path = "/content/cam6.avi"
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Setup Video Writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('fall_detection_output4.avi', fourcc, fps, (frame_width, frame_height))

CONFIDENCE_THRESHOLD = 0.8  # Higher threshold to avoid false person detections

print("Processing video for fall detection...")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    pil_img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    img_tensor = transform(pil_img).unsqueeze(0).to(device)

    # Model Inference
    with torch.no_grad():
        outputs = model(img_tensor)

    # Filter by confidence threshold
    probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]
    keep = probas.max(-1).values > CONFIDENCE_THRESHOLD

    bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], (frame_width, frame_height))
    prob_keep = probas[keep]

    # Analyze boxes specifically for people and falls
    for p, (xmin, ymin, xmax, ymax) in zip(prob_keep, bboxes_scaled.tolist()):
        cl = p.argmax()

        # Only process if the detected object is a person
        if cl == PERSON_CLASS_IDX:
            w = xmax - xmin
            h = ymax - ymin

            # Calculate Aspect Ratio (Width / Height)
            aspect_ratio = w / h

            # Heuristic: A standing person has h > w (ratio < 1).
            # A fallen person typically has w > h (ratio > 1.1)
            if aspect_ratio > 1.1:
                label = f"FALL DETECTED! Ratio: {aspect_ratio:.2f}"
                box_color = (0, 0, 255)  # Red for Fall
                text_color = (0, 0, 255)
            else:
                label = "Person Standing"
                box_color = (0, 255, 0)  # Green for Normal
                text_color = (0, 255, 0)

            # Draw bounding box and alert text
            cv2.rectangle(frame, (int(xmin), int(ymin)), (int(xmax), int(ymax)), box_color, 3)
            cv2.putText(frame, label, (int(xmin), int(ymin) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, text_color, 2)

    out.write(frame)

cap.release()
out.release()
cv2.destroyAllWindows()
print("Finished! Saved as fall_detection_output.mp4")


Using cache found in /root/.cache/torch/hub/facebookresearch_detr_main


Processing video for fall detection...
Finished! Saved as fall_detection_output.mp4
